### Model Loading

In [ ]:
from mmdet.apis import init_detector, inference_detector
from mmengine.config import Config
from mmengine.runner import load_checkpoint
from mmcv.transforms import Compose

import torch

import sys

sys.path.append('/Data_large/marine/PythonProjects/MMDET/notebooks/Tools')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs/custom_components')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')



# Specify the path to model config and checkpoint file
config_file = 'config.py'
checkpoint_file = 'weights.pth'

# build the model from a config file and a checkpoint file
DetModel = init_detector(config_file, checkpoint_file, device='cuda:0')

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
cfg = Config.fromfile(config_file)
cfg = cfg.copy()
test_pipeline = cfg.test_dataloader.dataset.pipeline

model = DetModel
checkpoint = checkpoint_file
checkpoint = load_checkpoint(model, checkpoint, map_location='cpu')


checkpoint_meta = checkpoint.get('meta', {})
dataset_meta = checkpoint_meta['dataset_meta']['classes']
model.dataset_meta = dataset_meta

model.to(device)
model.eval()

In [ ]:
image_height = 512
image_width = 512
dummy_input = torch.randn(1, 1, image_height, image_width).to(device)  # Ensure dummy input is on the same device as the model

with torch.no_grad():
    output = model(dummy_input)

for o in output:
    for subo in o:
        print(subo.shape)

### torch2ONNX (mode1)

In [ ]:
image_height = 256+128

image_width = image_height
dummy_input = torch.randn(1, 1, image_height, image_width).to(device)  # Ensure dummy input is on the same device as the model

onnx_path = f'apisonnx/model_{image_height}.onnx'

torch.onnx.export(
                model,
                dummy_input,
                onnx_path,
            )
print(f"ONNX model exported to {onnx_path}.")

In [ ]:
!mo --input_model ./model.onnx --output_dir ./ --scale 1 --mean_values [0] --model_name end2end 

#### torch2ONNX (mode2): method that employs mmdeploy 

In [ ]:
# EXPORT TO ONNX USING DEPLOYER IN RUNSCRIPTS

In [ ]:
!mo --input_model ./export/end2end.onnx --output_dir ./export/ --scale 1 --mean_values [0] --compress_to_fp16 --model_name end2endIR

### torch2ONNX (mode3): method that employs from mmdeploy.apis.onnx import export 

In [ ]:
from mmdeploy.apis.onnx import export
import torch

image_height = 512
image_width = 512
dummy_input = torch.randn(1, 1, image_height, image_width).to(device)  # Ensure dummy input is on the same device as the model


export(model = model,
           args = dummy_input,
           output_path_prefix = '/Data_large/marine/PythonProjects/MMDET/notebooks/OpenVINO/apisonnx',
           backend = 'default',
           input_metas = None,
           context_info = dict(),
           input_names = None,
           output_names = None,
           opset_version = 11,
           dynamic_axes = None,
           verbose = False,
           keep_initializers_as_inputs = None,
           optimize = False)


In [ ]:
!mo -h